| Task | Input | Output |
|:-----|:----:|-----:|
| ATE    | S    | aspect_i or noaspect    |
| ATSC    | S + aspect   | sentiment_i   or none |
|ASPE   | S    | (asp_i, sentiment_i) or noaspect:none  |
|AOOE   | S + aspect  | opinion_i  or none  |
|AOPE   | S | (aspect_i, opinion_i) or noaspect:none |
|AOSTE | S | (aspect_i, opinion_i, sentiment_i) or noaspect:none:none |


In [1]:
DATA_PATH = "/kaggle/working/InstructABSA/data/refine/aos_refine.csv"

In [2]:
import json
import pandas as pd
from utils import *
from sklearn.model_selection import train_test_split
import ast

In [3]:
data = pd.read_csv(DATA_PATH)

In [4]:
df = format_data(data)

In [5]:
df

,text,aspects,aspect_sentiment_pairs,aspect_opinion_pairs,triplets
0,sản phẩm tốt trong tầm giá. da pu nhìn cũng đẹ...,"[implicit_aspect, da pu, sản phẩm]","[(sản phẩm, POS), (da pu, POS), (implicit_aspe...","[(sản phẩm, tốt trong tầm giá.), (da pu, đẹp.)...","[(sản phẩm, tốt trong tầm giá., POS), (da pu, ..."
1,làm ăn chán thế ad trả hàng 2 lần rồi mà vẫn g...,"[hàng, làm ăn, màu]","[(làm ăn, NEG), (hàng, NEG), (màu, NEG)]","[(làm ăn, chán), (hàng, lỗi), (màu, toàn đưa 1...","[(làm ăn, chán, NEG), (hàng, lỗi, NEG), (màu, ..."
2,Đã nhận hàng và rất ưng ý với sản phẩm. Giá tu...,"[Giá, sản phẩm., sản phẩm]","[(sản phẩm., POS), (Giá, POS), (sản phẩm, POS)]","[(sản phẩm., rất ưng ý), (Giá, rẻ), (sản phẩm,...","[(sản phẩm., rất ưng ý, POS), (Giá, rẻ, POS), ..."
3,"sp dễ nấu, hơi có mùi khét nếu k khuấy lên liê...","[nấu,, mùi, cháo]","[(nấu,, POS), (mùi, NEG), (cháo, POS)]","[(nấu,, dễ), (mùi, khét), (cháo, nhừ nhanh, hấ...","[(nấu,, dễ, POS), (mùi, khét, NEG), (cháo, nhừ..."
4,hộp sữa giao bị móp. cái này không phải do lỗi...,[hộp sữa],"[(hộp sữa, NEG)]","[(hộp sữa, bị móp.)]","[(hộp sữa, bị móp., NEG)]"
...,...,...,...,...,...
5355,"Bút viết đẹp, mực ra đều, giao hàng nhanh, tiế...","[viết, mực, giao hàng]","[(viết, POS), (mực, POS), (giao hàng, POS)]","[(viết, đẹp,), (mực, ra đều,), (giao hàng, nha...","[(viết, đẹp,, POS), (mực, ra đều,, POS), (giao..."
5356,"chất lượng sản phẩm tốt shop cực kì dễ thương,...","[chất lượng sản phẩm, shop, gọi điện tư vấn]","[(chất lượng sản phẩm, POS), (shop, POS), (gọi...","[(chất lượng sản phẩm, tốt), (shop, cực kì dễ ...","[(chất lượng sản phẩm, tốt, POS), (shop, cực k..."
5357,nhận hàng được tiki đóng gói kỹ. hình thú ghép...,"[implicit_aspect, hình thú ghép, đóng gói, bản...","[(đóng gói, POS), (hình thú ghép, POS), (impli...","[(đóng gói, kỹ.), (hình thú ghép, dễ thương,),...","[(đóng gói, kỹ., POS), (hình thú ghép, dễ thươ..."
5358,Kính chuẩn màn hình note10 plus . ôm cong sát ...,"[Kính, ôm cong]","[(Kính, POS), (ôm cong, POS)]","[(Kính, chuẩn màn hình note10 plus), (ôm cong,...","[(Kính, chuẩn màn hình note10 plus, POS), (ôm ..."


In [6]:
df.to_csv("/kaggle/working/InstructABSA/data/refine/raw.csv", index=False)

# Chọn vài mẫu ra làm example cho instruction


In [7]:
raw = pd.read_csv("/kaggle/working/InstructABSA/data/refine/raw.csv")

- Chia ra dựa theo sentiment.
- InstructABSA1 : 2 example POS
- InstructABSA2 : 2 POS, 2 NEU, 2 NEG
- Trong bài khảo sát : 1 JOINT (tức là 1 example nhưng có đủ cả POS NEU và NEG)

In [8]:
only_pos = [] # những câu chỉ toàn pos
only_neg = [] # những câu chỉ toàn neg
only_neu = [] # những câu chỉ toàn neu (ít khả năng)
joint = [] # những câu có pos và neg và neu

for i in range(len(raw)):
    as_pairs = ast.literal_eval(raw.iloc[i].aspect_sentiment_pairs)
    sentiments = []
    for pair in as_pairs:
        sentiments.append(pair[1])
    sentiments = set(sentiments)

    if len(sentiments) == 0:
        continue

    if len(sentiments) == 3:
        joint.append(i)
    if len(sentiments) == 1:
        if "NEU" in sentiments:
            only_neu.append(i)
        elif "POS" in sentiments:
            only_pos.append(i)
        elif "NEG" in sentiments:
            only_neg.append(i)


In [9]:
print(f"Only positive sentiment: {len(only_pos)}")
print(f"Only negative sentiment: {len(only_neg)}")
print(f"Only neutral sentiment: {len(only_neu)}")
print(f"Joint sentiment: {len(joint)}")

Only positive sentiment: 2795
Only negative sentiment: 728
Only neutral sentiment: 49
Joint sentiment: 203


In [10]:
import random

In [19]:
pos_example_idx =  random.sample(only_pos, 12)
print(f"pos_example_idx: {pos_example_idx}")
neg_example_idx = random.sample(only_neg, 12)
print(f"neg_example_idx: {neg_example_idx}")
neu_example_idx = random.sample(only_neu, 12)
print(f"neu_example_idx: {neu_example_idx}")
joint_example_idx = random.sample(joint, 12)
print(f"joint_example_idx: {joint_example_idx}")

pos_example_idx: [3129, 2793, 2399, 2434, 227, 5241, 5061, 2603, 904, 5068, 3707, 4128]
neg_example_idx: [3587, 3582, 4307, 3937, 3209, 828, 1310, 3650, 3311, 5022, 568, 2182]
neu_example_idx: [2187, 2485, 3815, 2233, 4140, 4486, 401, 5269, 475, 4649, 3160, 1638]
joint_example_idx: [1692, 2795, 4394, 5302, 4444, 3876, 5255, 2382, 4889, 636, 4794, 1038]


In [20]:
raw = pd.read_csv("/kaggle/working/InstructABSA/data/refine/raw.csv")

In [21]:
# tách ra làm các df con 
pos_example = raw.iloc[pos_example_idx].copy().reset_index(drop=True)
neg_example = raw.iloc[neg_example_idx].copy().reset_index(drop=True)
neu_example = raw.iloc[neu_example_idx].copy().reset_index(drop=True)
joint_example = raw.iloc[joint_example_idx].copy().reset_index(drop=True)
#
raw = raw.drop(pos_example_idx + neg_example_idx + neu_example_idx + joint_example_idx).reset_index(drop=True)

In [22]:
pos_example.to_csv("/kaggle/working/InstructABSA/data/refine/example/example_pos.csv", index=False)
neu_example.to_csv("/kaggle/working/InstructABSA/data/refine/example/example_neu.csv", index=False)
neg_example.to_csv("/kaggle/working/InstructABSA/data/refine/example/example_neg.csv", index=False)
joint_example.to_csv("/kaggle/working/InstructABSA/data/refine/example/example_joint.csv", index=False)

# Chia ra làm train test dev

In [23]:
def split_dataframe(
    df,
    train_ratio=0.8,
    dev_ratio=0.1,
    test_ratio=0.1,
    seed=42
):
    assert train_ratio + dev_ratio + test_ratio == 1.0

    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - train_ratio),
        random_state=seed
    )

    dev_df, test_df = train_test_split(
        temp_df,
        test_size=test_ratio / (dev_ratio + test_ratio),
        random_state=seed
    )

    return train_df, dev_df, test_df


In [24]:
train_df, dev_df, test_df = split_dataframe(raw)

In [25]:
train_df.to_csv("/kaggle/working/InstructABSA/data/refine/train.csv", index=False)
dev_df.to_csv("/kaggle/working/InstructABSA/data/refine/dev.csv", index=False)
test_df.to_csv("/kaggle/working/InstructABSA/data/refine/test.csv", index=False)

# Tạo example

In [26]:
import pandas as pd
from utils import *

In [55]:
example = pd.read_csv("/kaggle/working/InstructABSA/data/refine/example/example_joint.csv")

In [56]:
import random
indexes = list(range(12))
random.shuffle(indexes)
print(indexes)

[2, 11, 0, 8, 1, 6, 9, 7, 4, 10, 5, 3]


Tạo example cho từng file theo từng loại
- Đầu tiên có 6 task - mỗi task chọn ra 2 câu
- Trong 2 câu này, với những câu có nhiều output như aspe thì ở mỗi loại instruct ta lấy sample random

In [57]:
start =1

In [58]:
tasks = ["ate", "atsc", "aspe", "aooe", "aope", "aoste"]
for i, task in zip(range(0,12,2),tasks):
    print(f"##################### {task} #####################")
    task_idx = indexes[i:i+2]
    ex_id = start
    print(task_idx)
    for idx in task_idx:
        df = example.iloc[[idx]].copy()
        data = create_data_with_task(df)
        data_for_task = data[data['Task'] == task]
        if len(data_for_task) > 1:
            data_for_task = data_for_task.sample(n=1)
        print(f"Ví dụ {ex_id}:")
        print(f"input: {data_for_task.iloc[0].Input}")
        print(f"output: {data_for_task.iloc[0].Output}")
        ex_id +=1

##################### ate #####################
[2, 11]
Ví dụ 1:
input: Áo bị rách, chất vải mỏng nhưng với giá tiền thì phù hợp, size rộng, tôi nghĩ không cho giây vào sẽ đẹp hơn ( giây rất xấu không khác gì giây quần )    :)))))))))
output: [ate] giây ## giá tiền ## chất vải ## size ## Áo
Ví dụ 2:
input: chất lượng tạm. chỉ format FAT32 thì copy nhanh, chuẩn NTFS thì chậm rì
output: [ate] copy ## chất lượng ## chuẩn NTFS
##################### atsc #####################
[0, 8]
Ví dụ 1:
input: Tiki gói bọc cẩn thận nhưng sản phẩm thì bị gỉ. Với mức giá 22.000 đồng thì tạm chấp nhận. Mọi người mua kích cỡ này sẽ rất phù hợp trồng những cây để bàn như sen đá. ## Aspect: giá
output: [atsc] NEU
Ví dụ 2:
input: Tiki nên kiểm tra lại hàng hoá và cách đóng gói. Hàng nhận về bị bể mất một mảng lớn ở nắp như vậy. Chả biết hộp bị sẵn hay do vận chuyển. May là lần này không bị ảnh hưởng đến sản phẩm bên trong, chứ còn một số hàng hoá dễ vỡ khác thì như nào nữa. ## Aspect: Tiki
output: [atsc] NEU


In [2]:
from datasets import Dataset

In [6]:
df = pd.read_csv("/kaggle/working/InstructABSA/data/dev.csv")
df = create_data_with_task(df)
df

,Task,Input,Output,index_sentence
0,ate,"giao hàng nhanh sản phẩm tốt, giá tốt, mọi ngư...",[ate] giao hàng ## sản phẩm ## giá,3832
1,aooe,"giao hàng nhanh sản phẩm tốt, giá tốt, mọi ngư...",[aooe] nhanh,3832
2,aooe,"giao hàng nhanh sản phẩm tốt, giá tốt, mọi ngư...","[aooe] tốt,",3832
3,aooe,"giao hàng nhanh sản phẩm tốt, giá tốt, mọi ngư...","[aooe] tốt,",3832
4,aope,"giao hàng nhanh sản phẩm tốt, giá tốt, mọi ngư...","[aope] giao hàng $ nhanh ## sản phẩm $ tốt, ##...",3832
...,...,...,...,...
6771,aope,Cực kỳ hài lòng về sản phẩm của Tiki có thể ti...,[aope] sản phẩm của Tiki $ Cực kỳ hài lòng ## ...,6547
6772,atsc,Cực kỳ hài lòng về sản phẩm của Tiki có thể ti...,[atsc] POS,6547
6773,atsc,Cực kỳ hài lòng về sản phẩm của Tiki có thể ti...,[atsc] POS,6547
6774,aspe,Cực kỳ hài lòng về sản phẩm của Tiki có thể ti...,[aspe] sản phẩm của Tiki $ POS ## implicit_asp...,6547


In [13]:
def get_dataframe_with_instruction(df, instruction_handler):
    N = len(df)
    Input = []
    Output = []
    for i in range(N):
        task = df.iloc[i].Task
        text = df.iloc[i].Input
        output = df.iloc[i].Output
        instruction = instruction_handler.apply_instruct(text, task)

        Input.append(instruction)
        Output.append(output)
    
    return Dataset.from_dict({"input": Input, "output": Output})


In [14]:
instruction_handler = InstructionsHandler()
instruction_handler.load_instruction_2()

In [15]:
dataset = get_dataframe_with_instruction(df, instruction_handler)

In [22]:
from transformers import AutoTokenizer

In [24]:
tokenizer = AutoTokenizer.from_pretrained("VietAI/vit5-base")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

# NEW Data


In [1]:
import json
import pandas as pd
# from utils import *
from sklearn.model_selection import train_test_split
import ast

In [2]:
df = pd.read_csv("/kaggle/working/InstructABSA/data/refine/aos_refine.csv")

In [3]:
df

,text,label
0,sản phẩm tốt trong tầm giá. da pu nhìn cũng đẹ...,sản phẩm tốt trong tầm giá. da pu nhìn cũng đẹ...
1,làm ăn chán thế ad trả hàng 2 lần rồi mà vẫn g...,làm ăn chán thế ad trả hàng 2 lần rồi mà vẫn g...
2,Đã nhận hàng và rất ưng ý với sản phẩm. Giá tu...,Đã nhận hàng và rất ưng ý với sản phẩm. Giá tu...
3,"sp dễ nấu, hơi có mùi khét nếu k khuấy lên liê...","sp dễ nấu, hơi có mùi khét nếu k khuấy lên liê..."
4,hộp sữa giao bị móp. cái này không phải do lỗi...,hộp sữa giao bị móp. cái này không phải do lỗi...
...,...,...
5355,"Bút viết đẹp, mực ra đều, giao hàng nhanh, tiế...","Bút viết đẹp, mực ra đều, giao hàng nhanh, tiế..."
5356,"chất lượng sản phẩm tốt shop cực kì dễ thương,...","chất lượng sản phẩm tốt shop cực kì dễ thương,..."
5357,nhận hàng được tiki đóng gói kỹ. hình thú ghép...,nhận hàng được tiki đóng gói kỹ. hình thú ghép...
5358,Kính chuẩn màn hình note10 plus . ôm cong sát ...,Kính chuẩn màn hình note10 plus . ôm cong sát ...


In [4]:
df.iloc[0].label.split("####")

['sản phẩm tốt trong tầm giá. da pu nhìn cũng đẹp. đáng để mọi người mua',
 "[([0, 1], [2, 3, 4, 5], 'POS'), ([6, 7], [10], 'POS'), ([], [11, 12, 13, 14, 15], 'POS')]"]

In [23]:
def format_sample(data, i):
  aspects = []
  aspect_sentiment_pairs = []
  aspect_opinion_pairs = []
  triplets = []

  parts = data.iloc[0].label.split("####")
  if len(parts) < 2:
    return {
  "text": parts[0],
  "aspects": aspects,
  "aspect_sentiment_pairs": aspect_sentiment_pairs,
  "aspect_opinion_pairs": aspect_opinion_pairs,
  "triplets": triplets
  }

  tokens = parts[0].split()
  raw_triplets = ast.literal_eval(parts[1])
  for raw_triplet in raw_triplets:
    a, o, p = raw_triplet
    if a:
      aspect = " ".join(tokens[a[0]: a[-1]+1])
    else:
      aspect = "implicit_aspect"

    if o:
      opinion = " ".join(tokens[o[0]: o[-1]+1])
    else:
      opinion = "no_opinion"

    sentiment = p

    aspects.append(aspect)
    aspect_sentiment_pairs.append((aspect, sentiment))
    aspect_opinion_pairs.append((aspect, opinion))
    triplets.append((aspect, opinion, sentiment))

  sample_new_format = {
  "text": parts[0],
  "aspects": list(set(aspects)),
  "aspect_sentiment_pairs": aspect_sentiment_pairs,
  "aspect_opinion_pairs": aspect_opinion_pairs,
  "triplets": triplets
  }
  return sample_new_format

In [24]:
format_sample(df,0)

{'text': 'sản phẩm tốt trong tầm giá. da pu nhìn cũng đẹp. đáng để mọi người mua',
 'aspects': ['da pu', 'sản phẩm', 'implicit_aspect'],
 'aspect_sentiment_pairs': [('sản phẩm', 'POS'),
  ('da pu', 'POS'),
  ('implicit_aspect', 'POS')],
 'aspect_opinion_pairs': [('sản phẩm', 'tốt trong tầm giá.'),
  ('da pu', 'đẹp.'),
  ('implicit_aspect', 'đáng để mọi người mua')],
 'triplets': [('sản phẩm', 'tốt trong tầm giá.', 'POS'),
  ('da pu', 'đẹp.', 'POS'),
  ('implicit_aspect', 'đáng để mọi người mua', 'POS')]}

# Segment data


In [7]:
DATA_PATH= "/kaggle/working/InstructABSA/data/segment/segment_refine.csv"

In [5]:
import json
import pandas as pd
from utils import *
from sklearn.model_selection import train_test_split
import ast

In [11]:
data = pd.read_csv(DATA_PATH)

In [12]:
data

,Unnamed: 0,text,postag,head,deprel,label
0,0,"Nhan hàng dung hẹn , hop gọi ky , nhân_viên gi...","['V', 'N', 'V', 'V', 'CH', 'N', 'V', 'N', 'CH'...","[0, 1, 2, 3, 1, 7, 1, 7, 1, 1, 10, 11, 14, 1]","['root', 'dob', 'nmod', 'vmod', 'punct', 'sub'...","Nhan hàng dung hẹn , hop gọi ky , nhân_viên gi..."
1,1,"Giao hàng đúng hạn , giá_cả hợp_lý . Lần sau s...","['V', 'N', 'A', 'N', 'CH', 'N', 'A', 'CH', 'N'...","[0, 1, 2, 3, 2, 2, 6, 1, 4, 1, 4, 0, 4, 4, 4]","['root', 'dob', 'nmod', 'amod', 'punct', 'nmod...","Giao hàng đúng hạn , giá_cả hợp_lý . Lần sau s..."
2,2,"sản_phẩm y hình , giao hàng nhanh , đóng_gói k...","['N', 'P', 'N', 'CH', 'V', 'N', 'A', 'CH', 'V'...","[13, 1, 1, 1, 1, 5, 6, 5, 5, 9, 5, 13, 0, 13]","['sub', 'det', 'nmod', 'punct', 'nmod', 'dob',...","sản_phẩm y hình , giao hàng nhanh , đóng_gói k..."
3,3,"mì thơm ngon , giá_cả hợp_lý , mới sản_xuất , ...","['N', 'A', 'A', 'CH', 'N', 'A', 'CH', 'R', 'V'...","[0, 1, 2, 1, 1, 5, 1, 9, 1, 1, 1, 11]","['root', 'nmod', 'amod', 'punct', 'nmod', 'nmo...","mì thơm ngon , giá_cả hợp_lý , mới sản_xuất , ..."
4,4,Rất ngon . Tớ rất thích ăn omachi . Sẽ ủng_hộ ...,"['R', 'A', 'CH', 'P', 'R', 'V', 'V', 'N', 'CH'...","[2, 0, 2, 3, 3, 0, 3, 4, 3, 2, 0, 2, 2]","['amod', 'root', 'punct', 'sub', 'adv', 'root'...",Rất ngon . Tớ rất thích ăn omachi . Sẽ ủng_hộ ...
...,...,...,...,...,...,...
5246,5528,Chất_lượng tốt giao hàng đúng hẹn đóng_gói kỹ_...,"['N', 'A', 'V', 'N', 'A', 'N', 'V', 'A']","[3, 1, 0, 3, 4, 5, 6, 3]","['sub', 'nmod', 'root', 'dob', 'nmod', 'amod',...",Chất_lượng tốt giao hàng đúng hẹn đóng_gói kỹ_...
5247,5529,Sp đẹp lắm shop . Để mua thêm mẫu khác . Nhà a...,"['N', 'A', 'T', 'Nb', 'CH', 'E', 'V', 'V', 'N'...","[2, 0, 2, 2, 2, 0, 1, 2, 2, 4, 1, 4, 4, 4, 0, 4]","['sub', 'root', 'amod', 'amod', 'punct', 'root...",Sp đẹp lắm shop . Để mua thêm mẫu khác . Nhà a...
5248,5530,Đặt màu xám mà giao màu này . Hàng_không trầy_...,"['V', 'N', 'A', 'C', 'V', 'N', 'P', 'CH', 'N',...","[0, 1, 2, 1, 4, 5, 6, 1, 0, 1, 1]","['root', 'dob', 'nmod', 'coord', 'conj', 'dob'...",Đặt màu xám mà giao màu này . Hàng_không trầy_...
5249,5531,"Chuột dùng rất ổn , chất_lượng tốt so với tầm ...","['N', 'V', 'R', 'A', 'CH', 'N', 'A', 'V', 'E',...","[2, 0, 4, 2, 2, 8, 6, 2, 8, 9, 10, 2]","['sub', 'root', 'amod', 'vmod', 'punct', 'sub'...","Chuột dùng rất ổn , chất_lượng tốt so với tầm ..."


In [9]:
df = format_data(data)

In [10]:
df

,text,aspects,aspect_sentiment_pairs,aspect_opinion_pairs,triplets
0,"Nhan hàng dung hẹn , hop gọi ky , nhân_viên gi...","[Nhan hàng, nhân_viên giao hàng, hop gọi]","[(Nhan hàng, POS), (hop gọi, POS), (nhân_viên ...","[(Nhan hàng, dung hẹn), (hop gọi, ky), (nhân_v...","[(Nhan hàng, dung hẹn, POS), (hop gọi, ky, POS..."
1,"Giao hàng đúng hạn , giá_cả hợp_lý . Lần sau s...","[Giao hàng, Tiki, giá_cả]","[(Giao hàng, POS), (giá_cả, POS), (Tiki, POS)]","[(Giao hàng, đúng hạn), (giá_cả, hợp_lý), (Tik...","[(Giao hàng, đúng hạn, POS), (giá_cả, hợp_lý, ..."
2,"sản_phẩm y hình , giao hàng nhanh , đóng_gói k...","[implicit_aspect, sản_phẩm, đóng_gói, giao hàng]","[(sản_phẩm, POS), (giao hàng, POS), (đóng_gói,...","[(sản_phẩm, y hình), (giao hàng, nhanh), (đóng...","[(sản_phẩm, y hình, POS), (giao hàng, nhanh, P..."
3,"mì thơm ngon , giá_cả hợp_lý , mới sản_xuất , ...","[implicit_aspect, mì, giá_cả, date]","[(mì, POS), (giá_cả, POS), (implicit_aspect, P...","[(mì, thơm ngon), (giá_cả, hợp_lý), (implicit_...","[(mì, thơm ngon, POS), (giá_cả, hợp_lý, POS), ..."
4,Rất ngon . Tớ rất thích ăn omachi . Sẽ ủng_hộ ...,"[implicit_aspect, omachi]","[(implicit_aspect, POS), (omachi, POS), (impli...","[(implicit_aspect, Rất ngon), (omachi, rất thí...","[(implicit_aspect, Rất ngon, POS), (omachi, rấ..."
...,...,...,...,...,...
5246,Chất_lượng tốt giao hàng đúng hẹn đóng_gói kỹ_...,"[Chất_lượng, giao hàng, đóng_gói]","[(Chất_lượng, POS), (giao hàng, POS), (đóng_gó...","[(Chất_lượng, tốt), (giao hàng, đúng hẹn), (đó...","[(Chất_lượng, tốt, POS), (giao hàng, đúng hẹn,..."
5247,Sp đẹp lắm shop . Để mua thêm mẫu khác . Nhà a...,"[Sp, implicit_aspect]","[(Sp, POS), (implicit_aspect, POS)]","[(Sp, đẹp lắm), (implicit_aspect, thích)]","[(Sp, đẹp lắm, POS), (implicit_aspect, thích, ..."
5248,Đặt màu xám mà giao màu này . Hàng_không trầy_...,"[implicit_aspect, Hàng_không]","[(implicit_aspect, NEG), (Hàng_không, POS)]","[(implicit_aspect, Đặt màu xám mà giao màu này...","[(implicit_aspect, Đặt màu xám mà giao màu này..."
5249,"Chuột dùng rất ổn , chất_lượng tốt so với tầm ...","[dùng, tầm tiền, chất_lượng]","[(dùng, POS), (chất_lượng, POS), (tầm tiền, POS)]","[(dùng, ổn), (chất_lượng, no_opinion), (tầm ti...","[(dùng, ổn, POS), (chất_lượng, no_opinion, POS..."


In [13]:
df.to_csv("/kaggle/working/InstructABSA/data/segment/raw.csv", index=False)

## Chọn example

In [20]:
raw = pd.read_csv("/kaggle/working/InstructABSA/data/segment/raw.csv")

In [21]:
only_pos = [] # những câu chỉ toàn pos
only_neg = [] # những câu chỉ toàn neg
only_neu = [] # những câu chỉ toàn neu (ít khả năng)
joint = [] # những câu có pos và neg và neu

for i in range(len(raw)):
    as_pairs = ast.literal_eval(raw.iloc[i].aspect_sentiment_pairs)
    sentiments = []
    for pair in as_pairs:
        sentiments.append(pair[1])
    sentiments = set(sentiments)

    if len(sentiments) == 0:
        continue

    if len(sentiments) == 3:
        joint.append(i)
    if len(sentiments) == 1:
        if "NEU" in sentiments:
            only_neu.append(i)
        elif "POS" in sentiments:
            only_pos.append(i)
        elif "NEG" in sentiments:
            only_neg.append(i)

print(f"Only positive sentiment: {len(only_pos)}")
print(f"Only negative sentiment: {len(only_neg)}")
print(f"Only neutral sentiment: {len(only_neu)}")
print(f"Joint sentiment: {len(joint)}")


Only positive sentiment: 2743
Only negative sentiment: 719
Only neutral sentiment: 48
Joint sentiment: 190


In [22]:
import random

In [23]:
pos_example_idx =  random.sample(only_pos, 12)
print(f"pos_example_idx: {pos_example_idx}")
neg_example_idx = random.sample(only_neg, 12)
print(f"neg_example_idx: {neg_example_idx}")
neu_example_idx = random.sample(only_neu, 12)
print(f"neu_example_idx: {neu_example_idx}")
joint_example_idx = random.sample(joint, 12)
print(f"joint_example_idx: {joint_example_idx}")

pos_example_idx: [1297, 5166, 3051, 201, 5229, 5163, 186, 3194, 1786, 5115, 2880, 1001]
neg_example_idx: [4508, 4207, 4983, 3407, 1718, 4919, 2786, 553, 2180, 2236, 1947, 93]
neu_example_idx: [5014, 2780, 4375, 2816, 5066, 4758, 239, 2999, 5005, 3187, 81, 2921]
joint_example_idx: [2356, 3607, 4836, 4264, 1904, 2782, 1285, 1387, 4807, 154, 3465, 4456]


In [24]:
# tách ra làm các df con 
pos_example = raw.iloc[pos_example_idx].copy().reset_index(drop=True)
neg_example = raw.iloc[neg_example_idx].copy().reset_index(drop=True)
neu_example = raw.iloc[neu_example_idx].copy().reset_index(drop=True)
joint_example = raw.iloc[joint_example_idx].copy().reset_index(drop=True)
#
raw = raw.drop(pos_example_idx + neg_example_idx + neu_example_idx + joint_example_idx ).reset_index(drop=True)

In [25]:
pos_example.to_csv("/kaggle/working/InstructABSA/data/segment/example/example_pos.csv", index=False)
neu_example.to_csv("/kaggle/working/InstructABSA/data/segment/example/example_neu.csv", index=False)
neg_example.to_csv("/kaggle/working/InstructABSA/data/segment/example/example_neg.csv", index=False)
joint_example.to_csv("/kaggle/working/InstructABSA/data/segment/example/example_joint.csv", index=False)


## Chia train test dev

In [26]:
def split_dataframe(
    df,
    train_ratio=0.8,
    dev_ratio=0.1,
    test_ratio=0.1,
    seed=42
):
    assert train_ratio + dev_ratio + test_ratio == 1.0

    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - train_ratio),
        random_state=seed
    )

    dev_df, test_df = train_test_split(
        temp_df,
        test_size=test_ratio / (dev_ratio + test_ratio),
        random_state=seed
    )

    return train_df, dev_df, test_df


In [27]:
train_df, dev_df, test_df = split_dataframe(raw)

In [28]:
train_df.to_csv("/kaggle/working/InstructABSA/data/segment/train.csv", index=False)
dev_df.to_csv("/kaggle/working/InstructABSA/data/segment/dev.csv", index=False)
test_df.to_csv("/kaggle/working/InstructABSA/data/segment/test.csv", index=False)

## Tạo example 

In [29]:
import pandas as pd
from utils import *

In [50]:
example = pd.read_csv("/kaggle/working/InstructABSA/data/segment/example/example_joint.csv")

In [51]:
import random
indexes = list(range(12))
random.shuffle(indexes)
print(indexes)

[2, 5, 3, 8, 1, 7, 6, 4, 10, 9, 11, 0]


In [52]:
start =1

In [53]:
tasks = ["ate", "atsc", "aspe", "aooe", "aope", "aoste"]
for i, task in zip(range(0,12,2),tasks):
    print(f"##################### {task} #####################")
    task_idx = indexes[i:i+2]
    ex_id = start
    for idx in task_idx:
        df = example.iloc[[idx]].copy()
        data = create_data_with_task(df)
        data_for_task = data[data['Task'] == task]
        if len(data_for_task) > 1:
            data_for_task = data_for_task.sample(n=1)
        print(f"Ví dụ {ex_id}:")
        print(f"input: {data_for_task.iloc[0].Input}")
        print(f"output: {data_for_task.iloc[0].Output}")
        ex_id +=1

##################### ate #####################
Ví dụ 1:
input: Điện_thoại bị trầy dán vào thấy đỡ xấu hơn hẳn : ) ) . Tiện_dụng nhưng hơi nhỏ , bỏ được tiền không nhiều , nhưng tầm giá vậy thì không đòi_hỏi thêm
output: [ate] implicit_aspect ## tầm giá ## dán vào ## bỏ được tiền
Ví dụ 2:
input: đã mua sp trên tiki nhiều lần , hàng do tiki cung_cấp mình rất hài_lòng . lần này hàng k phải do tiki cung_cấp , sp rất tệ , nắp nồi bị hở , nên bay hết_hơi nước , nấu ít sẽ bị khô , cháy cháo . bảo_hành cho mình đi
output: [ate] implicit_aspect ## hàng do tiki cung_cấp ## sp ## bảo_hành ## nấu ## nắp nồi
##################### atsc #####################
Ví dụ 1:
input: Nhận được serum vài ngày rồi nhưng hôm_nay mới dùng . Giao hàng hơi lâu nhưng được_cái bọc hàng rất cẩn_thận , cầm hộp mới nguyên lên không tí sứt_mẻ nào đã thích rồi . Mình mua của HAFA BEAUTY , date cũng mới từ tháng 6 thôi , nhận hàng một cái là cho luôn vào tủ_lạnh , đến hôm_nay mới mang ra dùng . Serum dạng lỏng , còn màu đỏ